# 06 — Visualizing processed tag data

Interactive exploration of processed tag data with **sleep-state events** overlaid,
using `plot_tag_data_interactive`.

**Example deployment:** `2020-04-10_mian-002` — a juvenile northern elephant seal
(*Mirounga angustirostris*) carrying EEG/EOG/EMG, ECG, and a motion+depth tag.

> **Note on data size.** This deployment's `outputs/` folder holds a trimmed
> 3-hour demo slice (2020-04-12 09:00–12:00 local). The full 4.2-day recording
> lives alongside it in `outputs_untrimmed/`. Set `USE_UNTRIMMED = True` below
> to load the full deployment instead (~12 GB pkl, slow to load).


## 1. Setup and load the deployment

In [ ]:
# Import necessary pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.utils.event_manager import *
from pyologger.plot_data.plotter import *
from pyologger.io_operations.base_exporter import *
from pyologger.utils.data_manager import *

import os
import pickle
import pandas as pd

dataset_id = "mian-juv-nese_sleep_lml-ano_JKB"
deployment_id = "2020-04-10_mian-002"

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()

In [ ]:
# Load the DataReader object.
#
# The pkl is read directly (rather than via select_and_load_deployment) so we can
# point at either the trimmed demo slice or the full recording.
USE_UNTRIMMED = False  # True -> load the full 4.2-day deployment (~12 GB)

outputs_dirname = "outputs_untrimmed" if USE_UNTRIMMED else "outputs"
deployment_folder = os.path.join(config["paths"]["local_private_data"], dataset_id, deployment_id)
outputs_folder = os.path.join(deployment_folder, outputs_dirname)
pkl_path = os.path.join(outputs_folder, "data.pkl")

with open(pkl_path, "rb") as f:
    data_pkl = pickle.load(f)

# The trimmed pkl was written with a stale output_folder; repoint it at the
# folder we actually loaded from so any re-save lands in the right place.
data_pkl.output_folder = outputs_folder
data_pkl.deployment_folder = deployment_folder

param_manager = ParamManager(deployment_folder=deployment_folder, deployment_id=deployment_id)

timezone = data_pkl.deployment_info["Time Zone"]
print(f"Deployment : {deployment_id}")
print(f"Loaded from: {outputs_folder}")
print(f"Time zone  : {timezone}")

In [ ]:
# What signals are available, and over what span?
rows = []
for name, df in data_pkl.signal_data.items():
    if df is None or "datetime" not in df.columns or df.empty:
        continue
    rows.append({
        "signal": name,
        "samples": len(df),
        "channels": ", ".join(c for c in df.columns if c != "datetime"),
        "start": df["datetime"].iloc[0],
        "end": df["datetime"].iloc[-1],
    })

signal_overview = pd.DataFrame(rows).sort_values("samples", ascending=False).reset_index(drop=True)
signal_overview

## 2. Inspect the state events

Sleep states are stored in `data_pkl.event_data` as rows with `type == "state"`,
each carrying a `datetime` (state onset) and a `duration` in seconds. Two parallel
scoring schemes are present:

| Prefix | Granularity | States |
|---|---|---|
| `sleep-state_*` | detailed | active/quiet waking, LV-SWS, HV-SWS, certain/putative REM |
| `simple_sleep_state.*` | collapsed | active waking, quiet waking, SWS, REM |

`dive` events are also present and are keyed to the depth trace.

In [ ]:
# Every state event available in the loaded window
events = data_pkl.event_data
state_events = events[events["type"] == "state"].copy()

summary = (
    state_events.groupby("key")
    .agg(n_events=("key", "size"), total_seconds=("duration", "sum"))
    .sort_values("total_seconds", ascending=False)
)
summary["total_minutes"] = (summary["total_seconds"] / 60).round(1)
summary[["n_events", "total_minutes"]]

In [ ]:
# Other (non-state) event keys — these become point annotations rather than shaded spans
events[events["type"] != "state"]["key"].value_counts()

## 3. Define the annotations

**State annotations** shade a time span on a target signal row. **Note annotations**
mark individual points (e.g. detected heartbeats) with a symbol.

No colors are specified here. Each event key's color and its target signal both resolve
from `color_mappings.json` — the sleep keys are registered there against the project's
existing sleep palettes (`__sleep_palette_primary__`, `__simple_sleep_palette__`), and
`__event_targets__` routes the detailed scheme to the EEG row and the simplified scheme
to the depth row. Wildcards are matched against the event keys present in the data.


In [ ]:
# Keys resolve by wildcard; colors and target signals come from color_mappings.json.
state_annotations_detailed = {"sleep-state_*": {}}       # -> shaded on the EEG row
state_annotations_simple = {"simple_sleep_state.*": {}}  # -> shaded on the depth row

state_annotations = {**state_annotations_detailed, **state_annotations_simple}

# Point annotations for detected beats. A note is only drawn if its target signal is
# one of the signals actually plotted -- markers are pinned to the top of that signal's
# row as an event rug, so there is no row to attach them to otherwise.
notes_to_plot = {
    "heartbeat_auto_detect_accepted":  {"signal": "heart_rate",  "symbol": "triangle-up",   "color": "green"},
    "heartbeat_auto_detect_rejected":  {"signal": "heart_rate",  "symbol": "x",             "color": "red"},
    "strokebeat_auto_detect_accepted": {"signal": "stroke_rate", "symbol": "triangle-up",   "color": "green"},
    "dive":                            {"signal": "depth",       "symbol": "triangle-down", "color": "blue"},
}

# Show what the wildcards expand to, and the color each key resolves to.
from pyologger.plot_data.plotter import _expand_state_annotation_patterns, load_color_mapping

color_mapping = load_color_mapping(color_mapping_path)
available_keys = sorted(data_pkl.event_data["key"].astype(str).unique())

for key in _expand_state_annotation_patterns(state_annotations, available_keys):
    print(f"{key:42s} {color_mapping.get(key)}")

## 4. Full-window interactive plot

`plot_tag_data_interactive` returns a `plotly-resampler` figure: it renders a
downsampled view and re-fetches full-resolution data as you zoom, which is what
makes 2.7M-sample ECG usable in a notebook.

The `depth` row is set as the range selector, so the slider at the bottom shows the
dive trace — drag it to scrub the whole figure.

**EEG rows get two extras automatically:**

- A **magma spectrogram** row is inserted directly above the signal it is computed
  from. It is not EEG-specific — `spectrogram_channel` accepts any channel (`"eeg_p4"`,
  `"ecg"`, `"odba"`, …) and the owning signal is inferred, so the row lands above that
  signal's traces. Defaults to the first plotted EEG channel.
  - `spectrogram_range=(low_hz, high_hz)` sets the displayed band (default `(0, 30)`).
  - `spectrogram_contrast=(low_pct, high_pct)` sets the dB percentiles used as color
    limits, each in `[0, 100]` (default `(2, 98)`). Widening toward `(0, 100)` lowers
    contrast; narrowing (e.g. `(20, 80)`) raises it. Limits re-fit to whatever window
    you zoom into.
  - Computed at the native sampling rate with 2 s windows at 75% overlap
    (0.5 s time step, 0.5 Hz frequency resolution). Disable with
    `show_spectrogram=False`.
- EEG channels are **stacked 200 µV apart** and the row is clipped to
  `n_channels × 200 µV`, so every channel keeps a true amplitude scale instead of
  being autoscaled independently, and the row grows taller with more channels.
  Change the spacing with `eeg_channel_offset_uv=...`, or pass `None` to disable.


In [ ]:
# Plot the whole loaded window.
TARGET_SAMPLING_RATE = 10  # Hz for the initial (zoomed-out) render

fig = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "ecg", "heart_rate", "stroke_rate", "odba", "prh"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],  # all four derivations
        "prh": ["pitch", "roll"],
    },
    state_annotations=state_annotations,
    note_annotations=notes_to_plot,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_range_selector_channel="depth",
    spectrogram_channel="eeg_p4",       # which channel the spectrogram is built from
    spectrogram_range=(0, 10),          # (low_hz, high_hz) band to display
    spectrogram_contrast=(2, 98),       # (low_pct, high_pct) dB percentiles for color limits
)

fig

### Choosing the spectrogram channel, band, and contrast

`spectrogram_channel` selects which channel the STFT is computed from — and because the
owning signal is inferred, the row is placed above whichever signal that channel belongs
to. The channel only has to exist in that signal's dataframe; it does **not** need to be
one of the plotted traces.

`spectrogram_range` narrows the band (sleep EEG is mostly under 10 Hz) and
`spectrogram_contrast` sets how tightly the magma ramp is fitted to the dB distribution.


In [ ]:
# Available EEG derivations for this deployment
print(data_pkl.signal_info["eeg"]["channels"])

# Compare two derivations over the delta/theta band, with tighter contrast.
for spec_ch in ["eeg_p4", "eeg_f3"]:
    fig_spec = plot_tag_data_interactive(
        data_pkl=data_pkl,
        signals=["depth", "eeg"],
        channels={"eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"]},
        state_annotations=state_annotations_simple,
        color_mapping_path=color_mapping_path,
        target_sampling_rate=TARGET_SAMPLING_RATE,
        zoom_range_selector_channel="depth",
        spectrogram_channel=spec_ch,
        spectrogram_range=(0, 10),      # delta + theta
        spectrogram_contrast=(10, 90),  # tighter than the (2, 98) default
    )
    fig_spec.show()

# The spectrogram is not EEG-only: any channel works, and the row follows its signal.
fig_odba_spec = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "odba"],
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_range_selector_channel="depth",
    spectrogram_channel="odba",
    spectrogram_range=(0, 2),           # stroke-band motion
)
fig_odba_spec.show()


## 5. Dedicated state-annotation channels

Shading alone can be hard to read when states are brief or overlapping. Passing
`state_annotation_channel_mode` adds compact rows that render each state as a solid
bar — a much clearer read of the sleep architecture.

- `"combined"` — one row holding every state
- `"separate"` — one row per state key

In [ ]:
# Same signals, but with the simplified states drawn as their own channel row.
fig_states = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "heart_rate", "odba"],
    channels={"eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"]},
    state_annotations=state_annotations_simple,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=TARGET_SAMPLING_RATE,
    zoom_range_selector_channel="depth",
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.15,
    render_state_on_signal_rows=False,   # bar row only; leaves the signals unshaded
)

fig_states

## 6. Zoom into a single sleep bout

Pass `time_range` to restrict the data loaded, and `zoom_start_time` /
`zoom_end_time` to open the figure already zoomed while keeping the surrounding
context available when you zoom back out.

In [ ]:
# Pick the longest REM bout in the window and center a view on it.
rem_events = state_events[state_events["key"].str.contains("rem", case=False, na=False)]

if not rem_events.empty:
    bout = rem_events.loc[rem_events["duration"].idxmax()]
    bout_start = bout["datetime"]
    bout_end = bout_start + pd.Timedelta(seconds=float(bout["duration"]))
    print(f"Longest REM bout: {bout['key']}")
    print(f"  {bout_start} -> {bout_end}  ({bout['duration']:.0f} s)")

    pad = pd.Timedelta(minutes=5)
    ZOOM_START, ZOOM_END = bout_start - pad, bout_end + pad
else:
    print("No REM events in this window; falling back to the first 30 minutes.")
    ZOOM_START = data_pkl.signal_data["depth"]["datetime"].iloc[0]
    ZOOM_END = ZOOM_START + pd.Timedelta(minutes=30)

In [ ]:
# High-resolution view of the bout: raw EEG/EOG/ECG rather than derived rates.
fig_bout = plot_tag_data_interactive(
    data_pkl=data_pkl,
    signals=["depth", "eeg", "eog", "ecg", "heart_rate"],
    channels={
        "eeg": ["eeg_p3", "eeg_p4", "eeg_f3", "eeg_f4"],
        "eog": ["eog_l", "eog_r"],
    },
    state_annotations=state_annotations_detailed,
    color_mapping_path=color_mapping_path,
    target_sampling_rate=100,          # higher rate for a short span
    zoom_start_time=ZOOM_START,
    zoom_end_time=ZOOM_END,
    zoom_range_selector_channel="depth",
    state_annotation_channel_mode="combined",
    state_annotation_channel_height_ratio=0.15,
)

fig_bout

## 7. Export

`plotly-resampler` figures are interactive in the notebook, but a saved HTML file is
static at the currently-rendered resolution. Export a modest time span so the written
file keeps usable detail.

In [ ]:
import plotly.offline as pyo

html_file_path = os.path.join(outputs_folder, f"{deployment_id}_sleep_states.html")

# Export the bout figure; .figure unwraps FigureResampler to a plain Figure.
fig_to_save = getattr(fig_bout, "figure", fig_bout)
pyo.plot(fig_to_save, filename=html_file_path, auto_open=False)

print(f"Wrote {html_file_path}")
print(f"  {os.path.getsize(html_file_path) / 1e6:.1f} MB")